In [5]:
import torch
print(torch.cuda.is_available())  # True
print(torch.cuda.get_device_name(0))  # NVIDIA GeForce RTX 3050

True
NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import timm
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import os
import numpy as np
from tqdm import tqdm  # Progress bars
import torch.onnx  # For ONNX export

In [7]:
# Transforms
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset
data_dir = '/home/rifat-cou/Documents/Project/Dataset_Raw'  # Your folder
full_dataset = datasets.ImageFolder(data_dir, transform=train_transform)
class_names = full_dataset.classes  # For classification_report

# 80/20 split
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])
val_dataset.dataset.transform = val_transform

# Loaders
batch_size = 16  # For 6GB VRAM; reduce if needed
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

num_classes = 5
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(device)
class_names = full_dataset.classes
print(f"Classes: {class_names}")
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

cuda
Classes: ['Chikenpox', 'Cowpox', 'Measles', 'MonkeyPox', 'Normal']
Train: 2088, Val: 523


In [8]:
# MobileNetV2
model1 = timm.create_model('mobilenetv2_100', pretrained=True, num_classes=num_classes)
for param in model1.blocks[:4].parameters():
    param.requires_grad = False
model1 = model1.to(device)

# EfficientNetB0
model2 = timm.create_model('efficientnet_b0', pretrained=True, num_classes=num_classes)
for param in model2.blocks[:4].parameters():
    param.requires_grad = False
model2 = model2.to(device)

# EfficientNetB1
model3 = timm.create_model('efficientnet_b1', pretrained=True, num_classes=num_classes)
for param in model3.blocks[:4].parameters():
    param.requires_grad = False
model3 = model3.to(device)

In [9]:
def train_single_model(model, epochs, save_name):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=1e-4)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.5)
    best_acc = 0.0
    unfreeze_epoch = epochs // 2
    train_losses = []
    lrs = []

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for inputs, labels in tqdm(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        avg_loss = train_loss / len(train_loader)
        train_losses.append(avg_loss)
        lrs.append(optimizer.param_groups[0]['lr'])
        
        # Validate
        model.eval()
        val_preds, val_labels = [], []
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                preds = torch.argmax(outputs, dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_labels.extend(labels.cpu().numpy())
        
        acc = accuracy_score(val_labels, val_preds)
        print(f'Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f} - Val Acc: {acc:.4f}')
        
        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), f'{save_name}_best.pth')
        
        scheduler.step()

        if epoch == unfreeze_epoch:
            for param in model.parameters():
                param.requires_grad = True
            optimizer = optim.AdamW(model.parameters(), lr=1e-5)
    
    # Save per-model curves
    plt.figure()
    plt.plot(range(1, epochs+1), train_losses)
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(f'Loss Curve - {save_name}')
    plt.savefig(f'{save_name}_loss_curve.png')
    plt.close()
    
    plt.figure()
    plt.plot(range(1, epochs+1), lrs)
    plt.xlabel('Epoch')
    plt.ylabel('LR')
    plt.title(f'LR Curve - {save_name}')
    plt.savefig(f'{save_name}_lr_curve.png')
    plt.close()

# Train each
train_single_model(model1, 200, 'mobilenetv2')
train_single_model(model2, 100, 'efficientnetb0')
train_single_model(model3, 200, 'efficientnetb1')

100%|█████████████████████████████████████████| 131/131 [00:12<00:00, 10.41it/s]


Epoch 1/200 - Loss: 1.3881 - Val Acc: 0.7744


100%|█████████████████████████████████████████| 131/131 [00:11<00:00, 11.43it/s]


Epoch 2/200 - Loss: 0.5439 - Val Acc: 0.8145


100%|█████████████████████████████████████████| 131/131 [00:11<00:00, 11.39it/s]


...

Epoch 198/200 - Loss: 0.0007 - Val Acc: 0.8834


100%|█████████████████████████████████████████| 131/131 [00:24<00:00,  5.43it/s]


Epoch 199/200 - Loss: 0.0010 - Val Acc: 0.8910


100%|█████████████████████████████████████████| 131/131 [00:24<00:00,  5.43it/s]


Epoch 200/200 - Loss: 0.0006 - Val Acc: 0.8815


In [13]:
# Load best models
model1.load_state_dict(torch.load('mobilenetv2_best.pth'))
model2.load_state_dict(torch.load('efficientnetb0_best.pth'))
model3.load_state_dict(torch.load('efficientnetb1_best.pth'))

models_list = [model1, model2, model3]

def ensemble_predict(models, inputs):
    with torch.no_grad():
        probs = [torch.softmax(m(inputs), dim=1) for m in models]
        avg_probs = torch.mean(torch.stack(probs), dim=0)
    return torch.argmax(avg_probs, dim=1), avg_probs

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        preds, probs = ensemble_predict(models_list, inputs)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average='weighted')
print(classification_report(all_labels, all_preds, target_names=class_names))
print(f'Final Val Accuracy: {acc:.4f} - F1 Score: {f1:.4f}')

model_path = '/home/rifat-cou/Documents/Project/Ensemble Models'
ensemble_dir = os.path.join(model_path, 'EfficientSubsetAverageEnsemble')
os.makedirs(ensemble_dir, exist_ok=True)
torch.save(model1.state_dict(), os.path.join(ensemble_dir, 'mobilenetv2.pth'))
torch.save(model2.state_dict(), os.path.join(ensemble_dir, 'efficientnetb0.pth'))
torch.save(model3.state_dict(), os.path.join(ensemble_dir, 'efficientnetb1.pth'))

# Plots (CM, ROC)
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8,6))
plt.imshow(cm, cmap='Blues')
plt.title('Confusion Matrix')
plt.colorbar()
plt.xticks(np.arange(num_classes), class_names, rotation=45)
plt.yticks(np.arange(num_classes), class_names)
plt.savefig(os.path.join(ensemble_dir, 'cm.png'))
plt.close()

all_labels_bin = label_binarize(all_labels, classes=range(num_classes))
plt.figure()
for i in range(num_classes):
    fpr, tpr, _ = roc_curve(all_labels_bin[:, i], np.array(all_probs)[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{class_names[i]} (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.title('ROC Curve')
plt.legend()
plt.savefig(os.path.join(ensemble_dir, 'roc.png'))
plt.close()

RuntimeError: Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same

In [15]:
# First, make sure all models are on the same device
models_list = [model.to(device) for model in models_list]  # Add this line to ensure all models are on the correct device

def ensemble_predict(models, inputs):
    with torch.no_grad():
        probs = [torch.softmax(m(inputs), dim=1) for m in models]
        avg_probs = torch.mean(torch.stack(probs), dim=0)
    return torch.argmax(avg_probs, dim=1), avg_probs

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        preds, probs = ensemble_predict(models_list, inputs)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average='weighted')
print(classification_report(all_labels, all_preds, target_names=class_names))
print(f'Final Val Accuracy: {acc:.4f} - F1 Score: {f1:.4f}')

model_path = '/home/rifat-cou/Documents/Project/Ensemble Models'
ensemble_dir = os.path.join(model_path, 'EfficientSubsetAverageEnsemble')
os.makedirs(ensemble_dir, exist_ok=True)
torch.save(model1.state_dict(), os.path.join(ensemble_dir, 'mobilenetv2.pth'))
torch.save(model2.state_dict(), os.path.join(ensemble_dir, 'efficientnetb0.pth'))
torch.save(model3.state_dict(), os.path.join(ensemble_dir, 'efficientnetb1.pth'))

# Plots (CM, ROC)
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8,6))
plt.imshow(cm, cmap='Blues')
plt.title('Confusion Matrix')
plt.colorbar()
plt.xticks(np.arange(num_classes), class_names, rotation=45)
plt.yticks(np.arange(num_classes), class_names)
plt.savefig(os.path.join(ensemble_dir, 'cm.png'))
plt.close()

all_labels_bin = label_binarize(all_labels, classes=range(num_classes))
plt.figure()
for i in range(num_classes):
    fpr, tpr, _ = roc_curve(all_labels_bin[:, i], np.array(all_probs)[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{class_names[i]} (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.title('ROC Curve')
plt.legend()
plt.savefig(os.path.join(ensemble_dir, 'roc.png'))
plt.close()

              precision    recall  f1-score   support

   Chikenpox       0.86      0.82      0.84       110
      Cowpox       0.98      0.97      0.98       104
     Measles       0.99      0.97      0.98        90
   MonkeyPox       0.87      0.88      0.87       104
      Normal       0.93      0.98      0.95       115

    accuracy                           0.92       523
   macro avg       0.92      0.92      0.92       523
weighted avg       0.92      0.92      0.92       523

Final Val Accuracy: 0.9216 - F1 Score: 0.9212


In [16]:
class EfficientSubsetAverageEnsembleONNX(nn.Module):
    def __init__(self, models_list):
        super().__init__()
        self.models = nn.ModuleList(models_list)

    def forward(self, x):
        probs = [torch.softmax(m(x), dim=1) for m in self.models]
        avg_probs = torch.mean(torch.stack(probs), dim=0)
        return avg_probs  # Return probs for softmax output

# Create wrapper
ensemble_model = EfficientSubsetAverageEnsembleONNX(models_list).to('cpu')

dummy_input = torch.randn(1, 3, 224, 224)
onnx_path = os.path.join(ensemble_dir, 'efficientsubsetaverageensemble.onnx')
torch.onnx.export(
    ensemble_model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=11,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)
print(f'Exported to ONNX: {onnx_path}')

Exported to ONNX: /home/rifat-cou/Documents/Project/Ensemble Models/EfficientSubsetAverageEnsemble/efficientsubsetaverageensemble.onnx
